# 📱 MONARCH STUDIO · phone-only notebook — m-test-01

**Aap ko sirf 3 tap karne hain:**

1. Upar menu → **Runtime** → **Change runtime type** → **T4 GPU** → Save
2. Upar menu → **Runtime** → **Run all**
3. 20–40 min baad neeche aakhri cell me `DONE` dikhega → mujhe chat me `DONE` likh do

Yeh notebook teeno kaam khud karta hai (jo stage chahiye, neeche toggle se ON/OFF):

| Stage | Kya banega | Free? |
|---|---|---|
| 1 · Clips | 1 silent clips, 480x832, Wan 2.1 1.3B | ✅ $0 |
| 2 · Voice | 0 line(s), Kokoro-82M (Apache-2.0) | ✅ $0 |
| 3 · Foley + Upscale | ambience/SFX + 480p → 960x1664 (Real-ESRGAN ×2) | ✅ $0 |

Output aap ke Drive folder me jayega:
`Wf-07 • Monarch Ai Agent / 05 · Projects/m-test-01/`
Agar Drive mount fail ho jaye (Colab ka auth masla) to notebook **local disk par** kaam karega aur aakhir me
ek **ZIP download** dega — kuch bhi rukta nahi.

> Mix, verify aur master main yahan se karunga — aap ko ffmpeg ya terminal ki zarurat **nahi**.


In [ ]:
# ── 1 · Setup (koi edit nahi) ────────────────────────────────────────────────
RUN_CLIPS = True       # silent clips (sabse lamba step)
RUN_VOICE = True       # Kokoro voice lines
RUN_FOLEY = True       # MMAudio ambience/SFX
RUN_UPSCALE = True     # 480p -> 2x
RUN_FINISH = True      # mix + MASTER (ffmpeg Colab me hi chalta hai)

import hashlib, json, pathlib, subprocess, time, zipfile

PROJECT = "m-test-01"
DRIVE_BASE = pathlib.Path(f"/content/drive/MyDrive/MONARCH STUDIO/Wf-07 • Monarch Ai Agent/05 · Projects/{PROJECT}")
LOCAL_BASE = pathlib.Path(f"/content/m_out/{PROJECT}")
DRIVE_OK = False

# Drive optional hai. Mount fail ho (credential propagation / 3rd-party cookies) to
# notebook local disk par kaam karta hai aur aakhir me ZIP download deta hai — kuch rukta nahi.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_BASE / ".write_test"
    probe.write_text("ok")
    probe.unlink()
    DRIVE_OK = True
    print("Drive  : mounted ✔")
except Exception as exc:
    print("Drive  : mount nahi hua —", type(exc).__name__, str(exc)[:110])
    print("         koi baat nahi: sab local disk par banega, aakhir me ZIP download milega")

BASE = DRIVE_BASE if DRIVE_OK else LOCAL_BASE
SHOTS = BASE / "shots"
DELIVERY = BASE / "delivery"
VO_DIR = BASE / "audio" / "vo"
FOLEY_DIR = BASE / "audio" / "foley"
for folder in (SHOTS, VO_DIR, FOLEY_DIR):
    folder.mkdir(parents=True, exist_ok=True)

JOBS = [
  {
    "id": "t01-plate",
    "seed": 42,
    "frames": 33,
    "steps": 30,
    "width": 480,
    "height": 832,
    "prompt": "cinematic medium shot of a single brass desk lamp on a dark walnut table, warm pool of light, soft volumetric haze, slow push in, shallow depth of field, film grain, no people, no text"
  }
]
VO_TEXT = {
  "t01-plate": ""
}
FOLEY_PROMPT = {
  "t01-plate": ""
}
FACTOR = 2

print("project :", BASE, "| exists:", BASE.exists())
print("shots   :", len(JOBS), "| voice:", len([v for v in VO_TEXT.values() if v.strip()]),
      "| foley:", len([v for v in FOLEY_PROMPT.values() if v.strip()]))

# Resume ka sasta raasta: clips + 2x pehle se Drive par hain to model dobara load na karo
# (3-5 min bachta hai). Sirf jo missing hai wohi stage chalta hai.
_pending = [job["id"] for job in JOBS
            if not ((SHOTS / f"{job['id']}.mp4").exists()
                    and (SHOTS / f"{job['id']}-{FACTOR}x.mp4").exists())]
if not _pending:
    RUN_CLIPS = RUN_UPSCALE = False
    print("clips   : pehle se Drive par ✔ — model load skip, seedha MASTER banta hai (~1 min)")
RUN_VOICE = RUN_VOICE and any(v.strip() for v in VO_TEXT.values())
RUN_FOLEY = RUN_FOLEY and any(v.strip() for v in FOLEY_PROMPT.values())


def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# ── 2 · Wan 2.1 T2V 1.3B load — Colab 12.7 GB RAM safe (4-bit UMT5) ─────────
pipe = None
if RUN_CLIPS:
    import subprocess as _sp, sys as _sys
    if _sys.version:
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.35",
                 "transformers>=4.49", "accelerate", "imageio", "imageio-ffmpeg", "ftfy",
                 "sentencepiece", "bitsandbytes"], check=False)

    import os, gc, ctypes, torch
    os.environ["HF_HUB_DISABLE_XET"] = "1"   # Xet downloader ka in-memory buffer hi Colab RAM todta hai
    try:                                     # glibc: freed pages OS ko turant wapas
        _libc = ctypes.CDLL("libc.so.6")
        _libc.mallopt(-3, 1 << 20)
        _libc.mallopt(-1, 64 << 10)
        _libc.mallopt(-8, 2)
        HAVE_TRIM = True
    except Exception:
        _libc = None
        HAVE_TRIM = False

    def host_ram():
        try:
            with open("/proc/meminfo") as fh:
                for line in fh:
                    if line.startswith("MemAvailable"):
                        return round(int(line.split()[1]) / 1048576, 1)
        except OSError:
            pass
        return None

    def free_mem(tag=""):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if HAVE_TRIM:
            _libc.malloc_trim(0)
        if tag:
            print("   · freed:", tag, "| host free:", host_ram(), "GB")

    from diffusers import AutoencoderKLWan, WanPipeline, WanTransformer3DModel
    from diffusers.schedulers import UniPCMultistepScheduler
    from transformers import UMT5EncoderModel, BitsAndBytesConfig as T5QuantConfig

    MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
    print("host RAM free:", host_ram(), "GB / 12.7")

    def build_pipe():
        # UMT5-XXL fp32 me 22.7 GB hai — 4-bit NF4 se ~2.4 GB VRAM, host RAM chhua hi nahi
        vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
        t5_quant = T5QuantConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16,
                                 llm_int8_enable_fp32_cpu_offload=True)
        text_encoder = UMT5EncoderModel.from_pretrained(
            MODEL_ID, subfolder="text_encoder", quantization_config=t5_quant,
            torch_dtype=torch.float16, device_map={"": 0})
        free_mem("text encoder 4-bit")
        transformer = WanTransformer3DModel.from_pretrained(
            MODEL_ID, subfolder="transformer", torch_dtype=torch.float16)
        free_mem("transformer fp16")
        built = WanPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=text_encoder,
                                            transformer=transformer, torch_dtype=torch.float16,
                                            low_cpu_mem_usage=True)
        built.transformer.to("cuda")
        built.vae.to("cuda")
        # 480p ke liye flow_shift 3.0 (5.0 sirf 720p) — warna texture muddy
        built.scheduler = UniPCMultistepScheduler(prediction_type="flow_prediction",
                                                  use_flow_sigmas=True, num_train_timesteps=1000,
                                                  flow_shift=3.0)
        built.vae.enable_tiling()
        built.vae.enable_slicing()
        free_mem("pipeline ready")
        return built

    try:
        pipe = build_pipe()
        print("pipeline ready ·", torch.cuda.get_device_name(0))
    except Exception as exc:
        pipe = None
        print("MODEL LOAD FAILED:", type(exc).__name__, str(exc)[:300])
        print("   yeh line chat me bhej do — main chhota config bana dunga")

# ── 3 · Clips (silent, 480x832) ─────────────────────────────────────────────
if RUN_CLIPS and pipe is not None:
    from diffusers.utils import export_to_video

    NEGATIVE = ("blurry, low quality, distorted, deformed hands, extra limbs, text overlay, "
                "watermark, jpeg artifacts, static frame, flicker")
    receipts = []
    started = time.time()
    for index, job in enumerate(JOBS, 1):
        out = SHOTS / f"{job['id']}.mp4"
        if out.exists() and out.stat().st_size > 10_000:
            print(f"[{index}/{len(JOBS)}] skip (already on Drive) {out.name}")
            continue
        print(f"[{index}/{len(JOBS)}] {job['id']}  {job['width']}x{job['height']} · {job['frames']} frames")
        t0 = time.time()
        try:
            frames = pipe(
                prompt=job["prompt"], negative_prompt=NEGATIVE,
                width=job["width"], height=job["height"], num_frames=job["frames"],
                num_inference_steps=job.get("steps", 18), guidance_scale=5.0,
                generator=torch.Generator("cpu").manual_seed(int(job.get("seed", 42))),
            ).frames[0]
            raw = pathlib.Path("/content/_raw_clip.mp4")
            export_to_video(frames, str(raw), fps=16)
            subprocess.run(["ffmpeg", "-y", "-i", str(raw), "-c:v", "libx264",
                            "-crf", "18", "-preset", "veryfast", "-pix_fmt", "yuv420p",
                            "-movflags", "+faststart", str(out)],
                           check=True, capture_output=True)
            del frames
            print(f"  ✔ {out.name}  {out.stat().st_size/1e6:.1f} MB · {md5(out)[:10]} · {time.time()-t0:.0f}s")
            receipts.append({"id": job["id"], "file": out.name, "md5": md5(out),
                             "seconds": round(time.time() - t0, 1)})
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print("  ✖ VRAM OOM — mujhe bolo, main frames/steps aur kam kar dunga")
            receipts.append({"id": job["id"], "error": "cuda OOM"})
        except Exception as exc:
            print("  ✖", type(exc).__name__, str(exc)[:200])
            receipts.append({"id": job["id"], "error": f"{type(exc).__name__}: {exc}"})
        free_mem(f"after {job['id']}")
        (BASE / "colab_clips_receipt.json").write_text(json.dumps(receipts, indent=2))
    del pipe
    pipe = None
    free_mem("Wan unloaded — upscale ke liye jagah")

# ── 4 · Voice — Kokoro-82M (Apache-2.0, GPU bhi nahi chahiye) ────────────────
if RUN_VOICE:
    import subprocess as _sp3, sys as _sys3
    _sp3.run([_sys3.executable, "-m", "pip", "install", "-q", "kokoro", "soundfile"], check=False)
    import numpy as np, soundfile as sf
    from kokoro import KPipeline

    pipeline = KPipeline(lang_code="a")      # 'a' American, 'b' British
    VOICE = "af_heart"                       # 54 options; badalna ho to mujhe bolo
    made = 0
    for shot_id, text in VO_TEXT.items():
        text = (text or "").strip()
        if not text:
            continue
        out = VO_DIR / f"{shot_id}.wav"
        if out.exists():
            print("skip (already on Drive)", out.name)
            continue
        chunks = [audio for _, _, audio in pipeline(text, voice=VOICE, speed=1.0)]
        audio = np.concatenate([c if isinstance(c, np.ndarray) else c.numpy() for c in chunks])
        sf.write(out, audio, 24000)
        print("vo", out.name, f"{len(audio)/24000:.2f}s")
        made += 1
    print("voice lines:", made)

# ── 5 · Foley / ambience — ambience bed (free, koi model download nahi) ─────
FOLEY_ENGINE = "mmaudio"     # bed | mmaudio | off
if RUN_FOLEY and FOLEY_ENGINE in ("bed", "mmaudio"):
    import re as _re5
    made = 0

    def shot_seconds(path):
        text = subprocess.run(["ffmpeg", "-hide_banner", "-i", str(path)],
                              capture_output=True, text=True).stderr
        found = _re5.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", text)
        if not found:
            return 5.0
        return int(found.group(1)) * 3600 + int(found.group(2)) * 60 + float(found.group(3))

    def bed_for(prompt):
        # prompt ke lafz se halka andaaza: barsaat/wind bright, sheher/crowd mid, warna room tone
        low = (prompt or "").lower()
        if any(word in low for word in ("rain", "wind", "storm", "water", "sea", "wave")):
            return "lowpass=f=5200,highpass=f=300"
        if any(word in low for word in ("crowd", "city", "traffic", "market", "street")):
            return "bandpass=f=900:width_type=q:w=0.7"
        return "lowpass=f=900,highpass=f=60"

    if FOLEY_ENGINE == "mmaudio":
        try:
            import subprocess as _sp4, sys as _sys4
            import torch, torchaudio
            from mmaudio.eval_utils import all_model_cfg, generate, load_video
            from mmaudio.model.flow_matching import FlowMatching
            from mmaudio.model.networks import get_my_mmaudio
            from mmaudio.model.utils.features_utils import FeaturesUtils

            vram = torch.cuda.get_device_properties(0).total_memory / 2**30
            cfg_name = "mmaudio_small_44k" if vram < 20 else "mmaudio_large_44k_v2"
            model_cfg = all_model_cfg[cfg_name]
            net = get_my_mmaudio(model_cfg.model).eval().to("cuda")
            feats = FeaturesUtils(**model_cfg.features).eval().to("cuda")
            fm = FlowMatching(min_sigma=0, inference_mode="euler", num_steps=25)
            net.load_weights(torch.load(f"/root/.cache/mmaudio/{cfg_name}.pth", map_location="cpu",
                                        weights_only=True))
            for job in JOBS:
                src = SHOTS / f"{job['id']}.mp4"
                out = FOLEY_DIR / f"{job['id']}.wav"
                if not src.exists() or out.exists():
                    continue
                info, video = load_video(str(src), 16, model_cfg.sampling_rate)
                with torch.no_grad():
                    audio = generate(video, feats, net, fm, num_steps=25,
                                     duration=float(info.duration), cfg_strength=4.5, seed=42)
                torchaudio.save(str(out), audio.cpu(), model_cfg.sampling_rate)
                print("foley(mmaudio)", out.name)
                made += 1
        except Exception as exc:
            print("MMAudio nahi chala (", type(exc).__name__, ") — ambience bed se aage badhta hoon")
            FOLEY_ENGINE = "bed"

    if FOLEY_ENGINE == "bed":
        for job in JOBS:
            src = SHOTS / f"{job['id']}.mp4"
            out = FOLEY_DIR / f"{job['id']}.wav"
            if not src.exists() or out.exists():
                continue
            try:
                dur = shot_seconds(src)
                fade_out = max(dur - 0.7, 0.0)
                chain = bed_for(FOLEY_PROMPT.get(job["id"], ""))
                subprocess.run(["ffmpeg", "-y", "-f", "lavfi",
                                "-i", f"anoisesrc=color=pink:duration={dur:.2f}:amplitude=0.2",
                                "-af", f"{chain},tremolo=f=0.3:d=0.25,afade=t=in:st=0:d=0.5,"
                                       f"afade=t=out:st={fade_out:.2f}:d=0.7",
                                "-ar", "48000", "-ac", "1", str(out)],
                               check=True, capture_output=True)
                print("foley(bed)", out.name, f"{dur:.2f}s")
                made += 1
            except Exception as exc:
                # koi bhi audio fail ho to bhi aage badho: master bina bed ke bhi ban jaye
                print("foley(bed) fail:", type(exc).__name__, str(exc)[:120])

    print("foley stems:", made)

# ── 6 · Upscale 480p → 960x1664 (Real-ESRGAN ×2, ffmpeg fallback) ──
if RUN_UPSCALE:
    import re as _re, shutil, subprocess, torch
    enhancer = None
    cv2 = None
    try:
        # Colab par basicsr ka wheel-build fail hota hai ("Getting requirements to build wheel"):
        # torch pehle se hai, is liye build isolation hata kar seedha legacy wheel install karo,
        # phir torchvision ke naye API ke liye functional_tensor import patch kar do.
        import subprocess as _sp2, sys as _sys2
        _sp2.run([_sys2.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
                  "basicsr==1.4.2", "realesrgan", "facexlib"], check=False)
        import pathlib as _pl, re as _re_patch
        _bs = None
        try:
            import basicsr, _pl as _p
            _bs = _p.Path(basicsr.__file__).parent
        except Exception:
            _bs = None
        if _bs is not None:
            for _f in _bs.rglob("*.py"):
                _txt = _f.read_text(errors="ignore")
                if "functional_tensor" in _txt:
                    _f.write_text(_txt.replace(
                        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
                        "from torchvision.transforms.functional import rgb_to_grayscale").replace(
                        "from torchvision.transforms.functional_tensor import", 
                        "from torchvision.transforms.functional import"))
        import cv2
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        enhancer = RealESRGANer(
            scale=4, half=True, tile=0, tile_pad=10, pre_pad=0, model=net,
            model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/RealESRGAN_x4plus.pth")
        print("upscaler: Real-ESRGAN ×2")
    except Exception as exc:
        enhancer = None
        print("Real-ESRGAN install/load nahi hua (", type(exc).__name__, ") — ffmpeg lanczos se ×2 karunga")

    def fps_of(src):
        text = subprocess.run(["ffmpeg", "-hide_banner", "-i", str(src)],
                              capture_output=True, text=True).stderr
        found = _re.search(r"(\d+(?:\.\d+)?)\s*fps", text)
        return found.group(1) if found else "16"

    def upscale_clip(src, dst, factor=FACTOR):
        if enhancer is None:
            subprocess.run(["ffmpeg", "-y", "-i", str(src), "-vf",
                            f"scale=iw*{factor}:ih*{factor}:flags=lanczos",
                            "-c:v", "libx264", "-crf", "17", "-pix_fmt", "yuv420p", str(dst)],
                           check=True, capture_output=True)
            return dst
        frames_in = pathlib.Path("/content/_up_in")
        frames_out = pathlib.Path("/content/_up_out")
        for folder in (frames_in, frames_out):
            shutil.rmtree(folder, ignore_errors=True)
            folder.mkdir(parents=True)
        subprocess.run(["ffmpeg", "-y", "-i", str(src), str(frames_in / "%06d.png")],
                       check=True, capture_output=True)
        for frame in sorted(frames_in.glob("*.png")):
            img = cv2.imread(str(frame), cv2.IMREAD_COLOR)
            big, _ = enhancer.enhance(img, outscale=factor)
            cv2.imwrite(str(frames_out / frame.name), big)
        subprocess.run(["ffmpeg", "-y", "-framerate", fps_of(src),
                        "-i", str(frames_out / "%06d.png"), "-c:v", "libx264", "-crf", "17",
                        "-pix_fmt", "yuv420p", str(dst)], check=True, capture_output=True)
        shutil.rmtree(frames_in, ignore_errors=True)
        shutil.rmtree(frames_out, ignore_errors=True)
        return dst

    made = 0
    for job in JOBS:
        src = SHOTS / f"{job['id']}.mp4"
        dst = SHOTS / f"{job['id']}-{FACTOR}x.mp4"
        if not src.exists() or dst.exists():
            continue
        print("upscaling", src.name)
        try:
            upscale_clip(src, dst)
            print("  ✔", dst.name, f"{dst.stat().st_size/1e6:.1f} MB")
            made += 1
        except Exception as exc:
            print("  ✖ upscale fail:", type(exc).__name__, str(exc)[:160], "(original clip safe hai)")
    print("upscaled:", made)

# ── 8 · Finish — per-shot mix + MASTER (ffmpeg Colab me hi chalta hai) ───────
if RUN_FINISH:
    import hashlib
    import json
    import pathlib
    import re
    import shutil
    import subprocess


    def md5(path: pathlib.Path) -> str:
        h = hashlib.md5()
        with path.open("rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()


    def probe(path: pathlib.Path, ffmpeg: str = "ffmpeg") -> dict:
        text = subprocess.run([ffmpeg, "-hide_banner", "-i", str(path)],
                              capture_output=True, text=True).stderr
        size = re.search(r"[,\s](\d{2,5})x(\d{2,5})(?=[\s,\[])", text)
        dur = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", text)
        fps = re.search(r"(\d+(?:\.\d+)?)\s*fps", text)
        acodec = re.search(r"Audio:\s*([a-z0-9_]+)", text)
        return {"name": path.name,
                "width": int(size.group(1)) if size else 0,
                "height": int(size.group(2)) if size else 0,
                "duration": round(int(dur.group(1)) * 3600 + int(dur.group(2)) * 60 + float(dur.group(3)), 2) if dur else 0.0,
                "fps": float(fps.group(1)) if fps else 0.0,
                "audio_codec": acodec.group(1) if acodec else None,
                "bytes": path.stat().st_size, "md5": md5(path)}


    def finish_master(shots, vo_dir, foley_dir, delivery, project, width, height,
                      fps=None, crf=18, loudnorm=True, ffmpeg="ffmpeg") -> dict:
        # Build one master from per-shot clips (+ optional voice/foley).
        # Per-shot pass, then concat, then loudness: each step is its own ffmpeg call so a
        # failure names the exact shot instead of dying inside one giant filtergraph.
        delivery = pathlib.Path(delivery)
        delivery.mkdir(parents=True, exist_ok=True)
        work = delivery / "_segments"
        work.mkdir(parents=True, exist_ok=True)
        segments, log = [], []

        for index, clip in enumerate(sorted(pathlib.Path(p) for p in shots), 1):
            clip = pathlib.Path(clip)
            if not clip.exists() or clip.stat().st_size < 1000:
                log.append(f"skip {clip.name} (missing/empty)")
                continue
            shot_id = re.sub(r"-(\d+)x$", "", clip.stem)
            info = probe(clip, ffmpeg)
            duration = info["duration"] or 2.0
            audio = []
            for folder in (vo_dir, foley_dir):
                candidate = pathlib.Path(folder) / f"{shot_id}.wav"
                if candidate.exists() and candidate.stat().st_size > 1000:
                    audio.append(candidate)

            seg = work / f"seg{index:02d}.mp4"
            cmd = [ffmpeg, "-y", "-i", str(clip)]
            for track in audio:
                cmd += ["-i", str(track)]
            # Wan native 16 fps deta hai; usay 24 par kheenchne se frames duplicate hote hain
            # aur video "stuck image" lagti hai. Is liye clip ka apna fps rakho.
            rate = fps or info["fps"] or 16.0
            vf = (f"scale={width}:{height}:force_original_aspect_ratio=decrease,"
                  f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={rate:g},"
                  f"format=yuv420p")
            if audio:
                if len(audio) > 1:
                    graph = "".join(f"[{i}:a]" for i in range(1, len(audio) + 1))
                    cmd += ["-filter_complex", f"{vf.split(',')[0] and ''}{graph}"
                            f"amix=inputs={len(audio)}:normalize=0[aout]"]
                    cmd += ["-map", "0:v", "-map", "[aout]"]
                    cmd += ["-vf", vf]
                else:
                    cmd += ["-map", "0:v", "-map", "1:a", "-vf", vf]
                cmd += ["-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2", "-shortest"]
            else:
                cmd += ["-f", "lavfi", "-t", f"{duration:.2f}", "-i",
                        "anullsrc=channel_layout=stereo:sample_rate=48000"]
                cmd += ["-map", "0:v", "-map", "1:a", "-vf", vf,
                        "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2", "-shortest"]
            cmd += ["-c:v", "libx264", "-crf", str(crf), "-preset", "veryfast",
                    "-pix_fmt", "yuv420p", str(seg)]
            proc = subprocess.run(cmd, capture_output=True, text=True)
            if proc.returncode != 0:
                log.append(f"FAILED segment {clip.name}: {' | '.join(proc.stderr.strip().splitlines()[-2:])}")
                continue
            segments.append(seg)
            log.append(f"segment {index}: {clip.name} + {len(audio)} audio -> {seg.name}")

        if not segments:
            raise RuntimeError("no usable clips: " + "; ".join(log))
        print(f"segments OK: {len(segments)}")

        # Concat demuxer ke andar path ko space par tod deta hai ("Wf-07 • Monarch Ai Agent"),
        # isliye list me sirf bare filename aur cwd = segments folder.
        listing = work / "list.txt"
        listing.write_text("".join("file " + seg.name + chr(10) for seg in segments))
        joined = work / "joined.mp4"
        proc = subprocess.run([ffmpeg, "-y", "-f", "concat", "-safe", "0", "-i", listing.name,
                               "-c", "copy", joined.name],
                              capture_output=True, text=True, cwd=str(work))
        if proc.returncode != 0:
            proc = subprocess.run([ffmpeg, "-y", "-f", "concat", "-safe", "0", "-i", listing.name,
                                   "-c:v", "libx264", "-crf", str(crf), "-c:a", "aac",
                                   "-b:a", "192k", joined.name],
                                  capture_output=True, text=True, cwd=str(work))
            if proc.returncode != 0:
                raise RuntimeError("concat failed: " + " | ".join(proc.stderr.strip().splitlines()[-2:]))

        print(f"concat OK: {joined.name} {joined.stat().st_size/1e6:.2f} MB")

        def silent(path, ffmpeg="ffmpeg") -> bool:
            # loudnorm digital silence par NaN deta hai -> AAC encoder "Qavg: nan" par marta hai.
            text = subprocess.run([ffmpeg, "-hide_banner", "-i", path.name,
                                   "-af", "volumedetect", "-f", "null", "-"],
                                  capture_output=True, text=True, cwd=str(path.parent)).stderr
            found = re.search(r"mean_volume:\s*(-?[\d.]+) dB", text)
            return (not found) or float(found.group(1)) <= -70.0

        master = delivery / f"{project}_MASTER.mp4"
        staged = work / f"{project}_MASTER.mp4"
        quiet = silent(joined, ffmpeg)
        if quiet:
            log.append("audio khaali hai - loudnorm skip (NaN se bachne ke liye)")
            loudnorm = False
        if loudnorm:
            cmd = [ffmpeg, "-y", "-i", joined.name, "-c:v", "copy",
                   "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "48000",
                   "-c:a", "aac", "-b:a", "192k",
                   "-movflags", "+faststart", staged.name]
            proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(work))
            if proc.returncode != 0 or not staged.exists() or staged.stat().st_size < joined.stat().st_size // 2:
                log.append("loudnorm nahi chala - master bina loudness pass ke")
                loudnorm = False
                shutil.copyfile(joined, staged)
        else:
            shutil.copyfile(joined, staged)
        shutil.copyfile(staged, master)
        print(f"master ok: {master.name} {master.stat().st_size/1e6:.2f} MB")

        info = probe(master, ffmpeg)
        receipt = {"project": project, "master": master.name, "shots": [s.name for s in segments],
                   "log": log, "info": info, "made_on": "colab-t4",
                   "loudnorm": "I=-16 TP=-1.5 LRA=11" if loudnorm else None}
        (pathlib.Path(delivery).parent / "colab_master_receipt.json").write_text(
            json.dumps(receipt, indent=2))
        return receipt
    try:
        _receipt = finish_master(
            [SHOTS / f"{job['id']}-{FACTOR}x.mp4" if (SHOTS / f"{job['id']}-{FACTOR}x.mp4").exists()
             else SHOTS / f"{job['id']}.mp4" for job in JOBS],
            VO_DIR, FOLEY_DIR, DELIVERY, PROJECT, 960, 1664, ffmpeg="ffmpeg")
        _info = _receipt["info"]
        print(f"MASTER OK  {_info['name']}  {_info['bytes']/1e6:.2f} MB  "
              f"{_info['width']}x{_info['height']}  {_info['duration']}s  "
              f"audio={_info['audio_codec']}  md5={_info['md5']}")
        print("master Drive me:", DELIVERY / _info["name"])
        for _line in _receipt["log"]:
            print("  ·", _line)
    except Exception as _exc:
        print("MASTER FAIL:", type(_exc).__name__, str(_exc)[:400])

# ── 7 · Summary + receipt (yeh cell hi "DONE" batata hai) ────────────────────
clips = sorted(SHOTS.glob("*.mp4"))
voice = sorted(VO_DIR.glob("*.wav"))
foley = sorted(FOLEY_DIR.glob("*.wav"))
rows = [{"file": f.name, "mb": round(f.stat().st_size / 1e6, 1), "md5": md5(f)}
        for f in clips + voice + foley]
(BASE / "colab_receipt.json").write_text(json.dumps(
    {"project": PROJECT, "clips": [c.name for c in clips],
     "voice": [v.name for v in voice], "foley": [f.name for f in foley],
     "files": rows}, indent=2))

print("clips :", len(clips))
print("voice :", len(voice))
print("foley :", len(foley))
for row in rows:
    print(f"  {row['file']:<28} {row['mb']:6.1f} MB  {row['md5'][:10]}")

zip_path = BASE / f"{PROJECT}_outputs.zip"   # Drive par hi likho: path har haal me valid hai
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    for file in clips + voice + foley:
        bundle.write(file, file.relative_to(BASE))
print()
print("ZIP    :", zip_path.name, f"{zip_path.stat().st_size/1e6:.1f} MB")
if DRIVE_OK:
    print("Drive  :", BASE)
    print("         ZIP:", zip_path.name)
else:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
        print("Download start hua — phone ke Downloads me ZIP ayega, woh mujhe chat me bhej do")
    except Exception as exc:
        print("download trigger nahi hua:", type(exc).__name__, exc)
        print("Left sidebar ke Files panel me ZIP maujood hai — wahan se bhi mil jayega")

ok = len([c for c in clips if not c.stem.endswith(("-2x", "-4x"))]) >= 1 and len(clips) >= 1
print()
if ok:
    print("DONE ✅  — mujhe chat me 'DONE' likh do, main mix + verify + master bana dunga")
else:
    print("ADHURA ⚠️  — jo missing hai mujhe bolo (main job chhota kar dunga)")
